# Applied Data Visualization and EDA with the UCI Iris Dataset

This executable notebook connects Saurav Singla's two companion articles:

1. **What are the important principles of data visualization?**
2. **Exploratory Data Analysis is a significant part of Data Science**

It uses Fisher's public Iris dataset through scikit-learn, so no live download or credential is required.

## Questions answered

- Is the dataset complete, balanced, and suitable for visual analysis?
- Which measurements differ most across species?
- Which variables are strongly related?
- When does interaction add value over a static chart?
- How can exploratory plots be refined into a concise evidence-based narrative?

## Reproducibility

Run from either the repository root or the `notebooks/` directory. Outputs are always written to the repository's `artifacts/` directory.


In [ ]:
from pathlib import Path
import platform

import matplotlib.pyplot as plt
import pandas as pd
import plotly
import plotly.express as px
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.datasets import load_iris

from dataviz_reference import apply_accessible_theme

WORKING_DIRECTORY = Path.cwd()
REPO_ROOT = (
    WORKING_DIRECTORY
    if (WORKING_DIRECTORY / "pyproject.toml").exists()
    else WORKING_DIRECTORY.parent
)
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory.")

ARTIFACTS = REPO_ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

apply_accessible_theme()
sns.set_context("notebook")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Plotly: {plotly.__version__}")
print(f"Artifacts: {ARTIFACTS.resolve()}")


## 1. Load and label the external dataset

The measurements are kept in centimetres and the numeric target is replaced with human-readable species names. Clear variable names and units reduce interpretation errors later.


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.rename(
    columns={
        "sepal length (cm)": "sepal_length_cm",
        "sepal width (cm)": "sepal_width_cm",
        "petal length (cm)": "petal_length_cm",
        "petal width (cm)": "petal_width_cm",
    }
)
df["species"] = df.pop("target").map(dict(enumerate(iris.target_names)))
features = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
]
display(df.head())


## 2. Inspect quality before plotting

EDA begins with structure, missingness, duplicates, cardinality, and summary statistics. The assertions make the notebook fail loudly if the bundled dataset changes unexpectedly.


In [ ]:
quality_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "unique_values": df.nunique(),
    }
)
display(quality_summary)
display(df.describe(include="all").T)

assert df.shape == (150, 5)
assert df[features].isna().sum().sum() == 0
assert set(df["species"]) == {"setosa", "versicolor", "virginica"}
assert df["species"].value_counts().nunique() == 1

print(f"Rows: {len(df):,}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print("Validation passed: complete, labelled, and balanced dataset.")


## 3. Category composition: use an honest magnitude encoding

A sorted horizontal bar chart supports category comparison. The scale starts at zero because bar length encodes magnitude, and direct labels remove unnecessary lookup.


In [ ]:
species_counts = df["species"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(species_counts.index, species_counts.values)
ax.set(
    title="The Iris dataset is balanced across species",
    xlabel="Number of observations",
    ylabel="",
    xlim=(0, species_counts.max() * 1.18),
)
ax.bar_label(bars, padding=4)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(ARTIFACTS / "iris_species_counts.png", dpi=160, bbox_inches="tight")
plt.show()


## 4. Distribution comparison: prefer small multiples over clutter

Each panel uses the same visual grammar while allowing its own numeric range. This makes it easier to see that petal measurements separate species more clearly than sepal measurements.


In [ ]:
long_df = df.melt(
    id_vars="species",
    value_vars=features,
    var_name="feature",
    value_name="measurement_cm",
)
grid = sns.displot(
    data=long_df,
    x="measurement_cm",
    hue="species",
    col="feature",
    col_wrap=2,
    kind="kde",
    fill=False,
    common_norm=False,
    height=3.2,
    facet_kws={"sharex": False, "sharey": False},
)
grid.set_axis_labels("Measurement (cm)", "Density")
grid.set_titles("{col_name}")
grid.figure.suptitle(
    "Petal measurements separate species more clearly than sepal measurements",
    y=1.03,
)
grid.figure.savefig(
    ARTIFACTS / "iris_feature_distributions.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 5. Relationship analysis: encode category with colour and shape

Both axes are quantitative, so a scatter plot is appropriate. Shape provides a second species cue, improving accessibility when colour distinctions are difficult to perceive.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.scatterplot(
    data=df,
    x="petal_length_cm",
    y="petal_width_cm",
    hue="species",
    style="species",
    s=75,
    alpha=0.8,
    ax=ax,
)
ax.set(
    title="Petal length and width reveal three distinct species groups",
    xlabel="Petal length (cm)",
    ylabel="Petal width (cm)",
)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.2)
ax.legend(title="Species", frameon=False)
fig.tight_layout()
fig.savefig(ARTIFACTS / "iris_petal_relationship.png", dpi=160, bbox_inches="tight")
plt.show()


## 6. Correlation: show values, not colour alone

A heatmap summarizes linear association but does not establish causation. Numeric annotations preserve exact values and make the chart usable without relying only on colour.


In [ ]:
correlation = df[features].corr()
petal_correlation = correlation.loc["petal_length_cm", "petal_width_cm"]
assert petal_correlation > 0.95

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Petal length and width have the strongest linear relationship")
fig.tight_layout()
fig.savefig(ARTIFACTS / "iris_correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Interaction only where it adds analytical value

Hover details allow inspection of individual flowers without permanently labelling every mark. The static chart remains the better choice for a concise report; the interactive chart is better for exploration.


In [ ]:
interactive = px.scatter(
    df,
    x="petal_length_cm",
    y="petal_width_cm",
    color="species",
    symbol="species",
    hover_data=["sepal_length_cm", "sepal_width_cm"],
    labels={
        "petal_length_cm": "Petal length (cm)",
        "petal_width_cm": "Petal width (cm)",
        "species": "Species",
    },
    title="Interactive Iris species comparison",
)
interactive.update_layout(
    template="plotly_white",
    legend_title_text="Species",
    hovermode="closest",
)
interactive.write_html(
    ARTIFACTS / "iris_interactive_scatter.html",
    include_plotlyjs="cdn",
)
interactive.show()


## 8. Turn exploration into a narrative

A useful conclusion is not that several charts were created. The evidence is:

- the dataset is complete and balanced;
- petal length and width differ substantially by species;
- those two features have a very strong positive linear relationship;
- petal measurements separate the species more clearly than sepal measurements;
- static visuals communicate the summary efficiently, while interaction supports record-level inspection.


In [ ]:
species_profile = (
    df.groupby("species", observed=True)[features]
    .agg(["mean", "median", "std"])
    .round(2)
)
display(species_profile)

expected_artifacts = {
    "iris_species_counts.png",
    "iris_feature_distributions.png",
    "iris_petal_relationship.png",
    "iris_correlation_heatmap.png",
    "iris_interactive_scatter.html",
}
created_artifacts = {path.name for path in ARTIFACTS.iterdir()}
missing_artifacts = expected_artifacts - created_artifacts
assert not missing_artifacts, f"Missing generated artifacts: {sorted(missing_artifacts)}"

print("Notebook completed successfully.")
print(f"Petal length/width correlation: {petal_correlation:.3f}")
print(f"Generated {len(expected_artifacts)} reproducible visualization artifacts.")


## Visual-design checklist used

- chart type selected from the analytical question;
- axes include measurement units;
- bar magnitude begins at zero;
- category is encoded with both colour and shape where appropriate;
- exact correlation values are annotated;
- decorative noise is reduced;
- titles state the finding rather than only naming the chart;
- interaction is used for inspection, not as decoration;
- conclusions distinguish correlation from causation.

## Dataset provenance and citation

- Dataset: Fisher's Iris dataset.
- Source: UCI Machine Learning Repository; distributed with scikit-learn.
- Access method: `sklearn.datasets.load_iris(as_frame=True)`.
- Network requirement: none.
- Generated outputs: repository `artifacts/` directory.
- For conceptual principles, cite the relevant Medium article in `ARTICLES.md`.
- For implementation, cite this repository release or commit hash.
